# 04a. Геофичи вокруг ЖК (OSM)

Считаем геопризнаки окружения каждого ЖК по координатам (`lat`/`lng`) через
OpenStreetMap (OSMnx 2.x). Фичи считаются **на уровне `project_id`** (один
репрезентативный центроид на проект, ~750 точек), а не на каждую из 4.6M строк —
это на три порядка дешевле, а гео-окружение проекта от среза к срезу не меняется.

Слои OSM кэшируются в `data/interim/osm_cache/*.gpkg`, поэтому повторный запуск
не дёргает Overpass. Результат → `data/processed/geo_features.parquet` (индекс
`project_id`); мерджится в модельный датасет в `04_features` по `project_id`.

In [1]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import display
pd.set_option("display.max_columns", 80)

_cwd = Path.cwd().resolve()
REPO_ROOT = _cwd if (_cwd / "data").is_dir() else _cwd.parent
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

import src.geo_features as geo
from src.config import EDA_LEFT_PARQUET, OSM_CACHE_DIR, GEO_FEATURES_PARQUET

df = pd.read_parquet(EDA_LEFT_PARQUET, columns=["project_id", "project_name", "lat", "lng"])
print(f"{len(df):,} строк, {df['project_id'].nunique()} уникальных project_id")

4,631,451 строк, 758 уникальных project_id


## 1. Точки проектов

Один центроид на `project_id` (median lat/lng). Координаты вне правдоподобных
границ Москвы/МО и нули (`0.0`) отбрасываются.

In [3]:
points = geo.project_points(df)
print(f"Проектов с валидными координатами: {len(points)}")
print(f"Отброшено проектов (нет/битые координаты): "
      f"{df['project_id'].nunique() - len(points)}")
display(points[["project_id", "project_name", "lat", "lng"]].head())

Проектов с валидными координатами: 752
Отброшено проектов (нет/битые координаты): 6


,project_id,project_name,lat,lng
0,551,Wellton Park,55.77341,37.47303
1,676,Union Park,55.78119,37.46896
2,683,Внуково Парк,55.59549,37.20044
3,688,Москва А101,55.56923,37.4905
4,696,Первый Московский,55.60198,37.36408


## 2. Скачивание слоёв OSM

Качаем потайлово (область режется на сетку); кэш по тайлам позволяет прерывать и продолжать — готовый слой берётся из кэша.

In [4]:
# накопитель слоёв; каждый слой ниже — отдельная ячейка (можно перезапускать по одному)
layers = {}
print("слои для скачивания:", list(geo.OSM_TAGS))

слои для скачивания: ['metro', 'schools', 'green', 'water', 'industrial', 'cemetery', 'power_lines', 'major_roads', 'railway']


**Точечные слои (тайл 10 км) — лёгкие.**

In [5]:
layers["metro"] = geo.fetch_layer(points, OSM_CACHE_DIR, "metro")

[metro] сетка ~10 км: 164 тайлов
  [metro] скачиваю 164 тайлов из OSM…
  [metro] тайл 1/164 | +5 (Σ5) |    0c | прошло  0.0м · ост.~ 0.1м
  [metro] тайл 2/164 | +2 (Σ7) |    0c | прошло  0.0м · ост.~ 0.1м
  [metro] тайл 3/164 | +0 (Σ7) |    0c | прошло  0.0м · ост.~ 0.1м
  [metro] тайл 4/164 | +4 (Σ11) |    0c | прошло  0.0м · ост.~ 0.0м
  [metro] тайл 5/164 | +10 (Σ21) |    0c | прошло  0.0м · ост.~ 0.0м
  [metro] тайл 6/164 | +6 (Σ27) |    0c | прошло  0.0м · ост.~ 0.0м
  [metro] тайл 7/164 | +7 (Σ34) |    0c | прошло  0.0м · ост.~ 0.0м
  [metro] тайл 8/164 | +16 (Σ50) |    0c | прошло  0.0м · ост.~ 0.0м
  [metro] тайл 9/164 | +6 (Σ56) |    0c | прошло  0.0м · ост.~ 0.0м
  [metro] тайл 10/164 | +70 (Σ126) |    0c | прошло  0.0м · ост.~ 0.0м
  [metro] тайл 11/164 | +13 (Σ139) |    0c | прошло  0.0м · ост.~ 0.0м
  [metro] тайл 12/164 | +27 (Σ166) |    0c | прошло  0.0м · ост.~ 0.0м
  [metro] тайл 13/164 | +9 (Σ175) |    0c | прошло  0.0м · ост.~ 0.0м
  [metro] тайл 14/164 | +56 (Σ231) 

In [12]:
layers["schools"] = geo.fetch_layer(points, OSM_CACHE_DIR, "schools")

[schools] сетка ~10 км: 164 тайлов
  [schools] скачиваю 164 тайлов из OSM…
  [schools] тайл 1/164 | +6 (Σ6) |    0c | прошло  0.0м · ост.~ 0.1м
  [schools] тайл 2/164 | +16 (Σ22) |    0c | прошло  0.0м · ост.~ 0.0м
  [schools] тайл 3/164 | +0 (Σ22) |    0c | прошло  0.0м · ост.~ 0.0м
  [schools] тайл 4/164 | +1 (Σ23) |    0c | прошло  0.0м · ост.~ 0.0м
  [schools] тайл 5/164 | +10 (Σ33) |    0c | прошло  0.0м · ост.~ 0.0м
  [schools] тайл 6/164 | +1 (Σ34) |    0c | прошло  0.0м · ост.~ 0.0м
  [schools] тайл 7/164 | +1 (Σ35) |    0c | прошло  0.0м · ост.~ 0.0м
  [schools] тайл 8/164 | +5 (Σ40) |    0c | прошло  0.0м · ост.~ 0.0м
  [schools] тайл 9/164 | +6 (Σ46) |    0c | прошло  0.0м · ост.~ 0.0м
  [schools] тайл 10/164 | +6 (Σ52) |    0c | прошло  0.0м · ост.~ 0.0м
  [schools] тайл 11/164 | +29 (Σ81) |    0c | прошло  0.0м · ост.~ 0.0м
  [schools] тайл 12/164 | +7 (Σ88) |    0c | прошло  0.0м · ост.~ 0.0м
  [schools] тайл 13/164 | +6 (Σ94) |    0c | прошло  0.0м · ост.~ 0.0м
  [school

**Тяжёлые площадные/линейные слои (тайл 5 км) — дольше.**

In [13]:
layers["green"] = geo.fetch_layer(points, OSM_CACHE_DIR, "green")

[green] сетка ~5 км: 482 тайлов
  [green] скачиваю 482 тайлов из OSM…
  [green] тайл 1/482 | +131 (Σ131) |    2c | прошло  0.0м · ост.~19.0м
  [green] тайл 2/482 | +124 (Σ255) |    2c | прошло  0.1м · ост.~19.0м
  [green] тайл 3/482 | +110 (Σ365) |    3c | прошло  0.1м · ост.~19.7м
  [green] тайл 4/482 | +108 (Σ473) |    3c | прошло  0.2м · ост.~20.3м
  [green] тайл 5/482 | +117 (Σ590) |    2c | прошло  0.2м · ост.~19.6м
  [green] тайл 6/482 | +54 (Σ644) |    2c | прошло  0.2м · ост.~19.3м
  [green] тайл 7/482 | +71 (Σ715) |    2c | прошло  0.3м · ост.~19.1м
  [green] тайл 8/482 | +156 (Σ871) |    3c | прошло  0.3м · ост.~19.2м
  [green] тайл 9/482 | +169 (Σ1,040) |    2c | прошло  0.4м · ост.~19.2м
  [green] тайл 10/482 | +106 (Σ1,146) |    2c | прошло  0.4м · ост.~19.1м
  [green] тайл 11/482 | +111 (Σ1,257) |    2c | прошло  0.4м · ост.~18.9м
  [green] тайл 12/482 | +257 (Σ1,514) |    3c | прошло  0.5м · ост.~19.0м
  [green] тайл 13/482 | +116 (Σ1,630) |    2c | прошло  0.5м · ост.~1

In [14]:
layers["water"] = geo.fetch_layer(points, OSM_CACHE_DIR, "water")

[water] сетка ~5 км: 482 тайлов
  [water] скачиваю 482 тайлов из OSM…
  [water] тайл 1/482 | +95 (Σ95) |    2c | прошло  0.0м · ост.~15.5м
  [water] тайл 2/482 | +65 (Σ160) |    2c | прошло  0.1м · ост.~14.8м
  [water] тайл 3/482 | +78 (Σ238) |    2c | прошло  0.1м · ост.~14.9м
  [water] тайл 4/482 | +78 (Σ316) |    2c | прошло  0.1м · ост.~15.1м
  [water] тайл 5/482 | +41 (Σ357) |    2c | прошло  0.2м · ост.~14.8м
  [water] тайл 6/482 | +36 (Σ393) |    2c | прошло  0.2м · ост.~14.6м
  [water] тайл 7/482 | +35 (Σ428) |    2c | прошло  0.2м · ост.~14.4м
  [water] тайл 8/482 | +60 (Σ488) |    2c | прошло  0.2м · ост.~14.3м
  [water] тайл 9/482 | +70 (Σ558) |    2c | прошло  0.3м · ост.~14.7м
  [water] тайл 10/482 | +47 (Σ605) |    2c | прошло  0.3м · ост.~14.9м
  [water] тайл 11/482 | +46 (Σ651) |    2c | прошло  0.3м · ост.~14.7м
  [water] тайл 12/482 | +58 (Σ709) |    2c | прошло  0.4м · ост.~14.8м
  [water] тайл 13/482 | +38 (Σ747) |    2c | прошло  0.4м · ост.~14.6м
  [water] тайл 14

In [15]:
layers["industrial"] = geo.fetch_layer(points, OSM_CACHE_DIR, "industrial")

[industrial] сетка ~5 км: 482 тайлов
  [industrial] скачиваю 482 тайлов из OSM…
  [industrial] тайл 1/482 | +9 (Σ9) |    2c | прошло  0.0м · ост.~12.4м
  [industrial] тайл 2/482 | +16 (Σ25) |    2c | прошло  0.1м · ост.~12.6м
  [industrial] тайл 3/482 | +6 (Σ31) |    2c | прошло  0.1м · ост.~12.8м
  [industrial] тайл 4/482 | +20 (Σ51) |    2c | прошло  0.1м · ост.~12.9м
  [industrial] тайл 5/482 | +15 (Σ66) |    2c | прошло  0.1м · ост.~13.0м
  [industrial] тайл 6/482 | +2 (Σ68) |    2c | прошло  0.2м · ост.~13.1м
  [industrial] тайл 7/482 | +0 (Σ68) |    1c | прошло  0.2м · ост.~11.8м
  [industrial] тайл 8/482 | +22 (Σ90) |    2c | прошло  0.2м · ост.~11.9м
  [industrial] тайл 9/482 | +8 (Σ98) |    2c | прошло  0.2м · ост.~12.0м
  [industrial] тайл 10/482 | +6 (Σ104) |    2c | прошло  0.3м · ост.~12.3м
  [industrial] тайл 11/482 | +5 (Σ109) |    2c | прошло  0.3м · ост.~12.3м
  [industrial] тайл 12/482 | +51 (Σ160) |    2c | прошло  0.3м · ост.~12.5м
  [industrial] тайл 13/482 | +2 (Σ

In [16]:
layers["cemetery"] = geo.fetch_layer(points, OSM_CACHE_DIR, "cemetery")

[cemetery] сетка ~10 км: 164 тайлов
  [cemetery] скачиваю 164 тайлов из OSM…
  [cemetery] тайл 1/164 | +2 (Σ2) |    1c | прошло  0.0м · ост.~ 3.9м
  [cemetery] тайл 2/164 | +4 (Σ6) |    1c | прошло  0.0м · ост.~ 3.7м
  [cemetery] тайл 3/164 | +1 (Σ7) |   30c | прошло  0.5м · ост.~28.9м
  [cemetery] тайл 4/164 | +1 (Σ8) |    2c | прошло  0.6м · ост.~22.9м
  [cemetery] тайл 5/164 | +6 (Σ14) |  180c | прошло  3.6м · ост.~113.5м
  [cemetery] тайл 6/164 | +2 (Σ16) |    2c | прошло  3.6м · ост.~94.7м
  [cemetery] тайл 7/164 | +4 (Σ20) |    1c | прошло  3.6м · ост.~81.2м
  [cemetery] тайл 8/164 | +1 (Σ21) |    2c | прошло  3.6м · ост.~71.1м
  [cemetery] тайл 9/164 | +6 (Σ27) |    2c | прошло  3.7м · ост.~63.4м
  [cemetery] тайл 10/164 | +3 (Σ30) |    6c | прошло  3.8м · ост.~58.1м
  [cemetery] тайл 11/164 | +5 (Σ35) |    1c | прошло  3.8м · ост.~52.8м
  [cemetery] тайл 12/164 | +0 (Σ35) |    0c | прошло  3.8м · ост.~48.2м
  [cemetery] тайл 13/164 | +4 (Σ39) |    1c | прошло  3.8м · ост.~44.5м

In [17]:
layers["major_roads"] = geo.fetch_layer(points, OSM_CACHE_DIR, "major_roads")

[major_roads] сетка ~5 км: 482 тайлов
  [major_roads] скачиваю 482 тайлов из OSM…
  [major_roads] тайл 1/482 | +11 (Σ11) |    2c | прошло  0.0м · ост.~14.1м
  [major_roads] тайл 2/482 | +21 (Σ32) |    2c | прошло  0.1м · ост.~14.7м
  [major_roads] тайл 3/482 | +5 (Σ37) |    2c | прошло  0.1м · ост.~14.5м
  [major_roads] тайл 4/482 | +21 (Σ58) |    2c | прошло  0.1м · ост.~14.5м
  [major_roads] тайл 5/482 | +171 (Σ229) |    2c | прошло  0.2м · ост.~14.6м
  [major_roads] тайл 6/482 | +0 (Σ229) |    1c | прошло  0.2м · ост.~13.1м
  [major_roads] тайл 7/482 | +2 (Σ231) |    2c | прошло  0.2м · ост.~13.2м
  [major_roads] тайл 8/482 | +37 (Σ268) |    2c | прошло  0.2м · ост.~13.3м
  [major_roads] тайл 9/482 | +15 (Σ283) |    2c | прошло  0.3м · ост.~13.4м
  [major_roads] тайл 10/482 | +0 (Σ283) |    1c | прошло  0.3м · ост.~12.6м
  [major_roads] тайл 11/482 | +38 (Σ321) |    2c | прошло  0.3м · ост.~12.7м
  [major_roads] тайл 12/482 | +60 (Σ381) |    2c | прошло  0.3м · ост.~12.9м
  [major_r

In [18]:
layers["railway"] = geo.fetch_layer(points, OSM_CACHE_DIR, "railway")

[railway] сетка ~5 км: 482 тайлов
  [railway] скачиваю 482 тайлов из OSM…
  [railway] тайл 1/482 | +28 (Σ28) |    1c | прошло  0.0м · ост.~11.0м
  [railway] тайл 2/482 | +0 (Σ28) |    0c | прошло  0.0м · ост.~ 6.8м
  [railway] тайл 3/482 | +0 (Σ28) |    0c | прошло  0.0м · ост.~ 5.3м
  [railway] тайл 4/482 | +50 (Σ78) |    1c | прошло  0.1м · ост.~ 6.9м
  [railway] тайл 5/482 | +0 (Σ78) |    0c | прошло  0.1м · ост.~ 5.9м
  [railway] тайл 6/482 | +0 (Σ78) |    0c | прошло  0.1м · ост.~ 5.4м
  [railway] тайл 7/482 | +0 (Σ78) |    0c | прошло  0.1м · ост.~ 4.9м
  [railway] тайл 8/482 | +13 (Σ91) |    1c | прошло  0.1м · ост.~ 5.7м
  [railway] тайл 9/482 | +2 (Σ93) |    1c | прошло  0.1м · ост.~ 6.2м
  [railway] тайл 10/482 | +0 (Σ93) |    0c | прошло  0.1м · ост.~ 5.8м
  [railway] тайл 11/482 | +0 (Σ93) |    0c | прошло  0.1м · ост.~ 5.5м
  [railway] тайл 12/482 | +59 (Σ152) |    1c | прошло  0.2м · ост.~ 5.9м
  [railway] тайл 13/482 | +0 (Σ152) |    0c | прошло  0.2м · ост.~ 5.7м
  [rai

In [19]:
layers["power_lines"] = geo.fetch_layer(points, OSM_CACHE_DIR, "power_lines")

[power_lines] сетка ~5 км: 482 тайлов
  [power_lines] скачиваю 482 тайлов из OSM…
  [power_lines] тайл 1/482 | +79 (Σ79) |    2c | прошло  0.0м · ост.~13.7м
  [power_lines] тайл 2/482 | +266 (Σ345) |    2c | прошло  0.1м · ост.~13.7м
  [power_lines] тайл 3/482 | +3 (Σ348) |    2c | прошло  0.1м · ост.~13.3м
  [power_lines] тайл 4/482 | +170 (Σ518) |    2c | прошло  0.1м · ост.~13.2м
  [power_lines] тайл 5/482 | +61 (Σ579) |    2c | прошло  0.1м · ост.~13.1м
  [power_lines] тайл 6/482 | +19 (Σ598) |    2c | прошло  0.2м · ост.~13.2м
  [power_lines] тайл 7/482 | +1 (Σ599) |    2c | прошло  0.2м · ост.~13.1м
  [power_lines] тайл 8/482 | +124 (Σ723) |    2c | прошло  0.2м · ост.~13.2м
  [power_lines] тайл 9/482 | +104 (Σ827) |    2c | прошло  0.3м · ост.~13.1м
  [power_lines] тайл 10/482 | +86 (Σ913) |    2c | прошло  0.3м · ост.~13.1м
  [power_lines] тайл 11/482 | +85 (Σ998) |    2c | прошло  0.3м · ост.~13.1м
  [power_lines] тайл 12/482 | +177 (Σ1,175) |    2c | прошло  0.3м · ост.~13.0м

### Сборка слоёв

Дочитываем все слои из кэша (готовые — мгновенно, без сети), чтобы `build_features`
получил полный набор даже если отдельные ячейки выше не запускались.

In [20]:
layers = geo.fetch_all_layers(points, OSM_CACHE_DIR)
print()
for name, gdf in layers.items():
    print(f"  {name:14} {len(gdf):>8,} объектов")

Сетка ~10 км: 164 тайлов
Сетка ~5 км: 482 тайлов

  metro            22,555 объектов
  schools           9,266 объектов
  green            99,377 объектов
  water            36,028 объектов
  industrial       22,285 объектов
  cemetery            913 объектов
  power_lines      46,941 объектов
  major_roads      40,495 объектов
  railway          19,550 объектов


## 3. Расчёт геофич

- **Расстояния** до ближайшего объекта: метро, школы, вода, промзона, кладбище, ЛЭП, крупная дорога, ж/д.
- **Счётчики** POI в кольцах (школы), **доли площади** (зелень/вода/промзона/кладбище),
  **длины** линий (дороги/ж/д/ЛЭП) и **плотность соседних ЖК** — по кольцам 500/1000/2000 м.

In [21]:
geo_feats = geo.build_features(points, layers, rings=geo.RINGS)
print(f"Геофич: {geo_feats.shape[1]} | проектов: {geo_feats.shape[0]}")
display(geo_feats.head())
print("\nОписательная статистика:")
display(geo_feats.describe().T[["mean", "min", "50%", "max"]].round(2))

Геофич: 35 | проектов: 752


,dist_to_metro_m,dist_to_school_m,dist_to_water_m,dist_to_industrial_m,dist_to_cemetery_m,dist_to_power_line_m,dist_to_major_road_m,dist_to_railway_m,schools_count_500m,schools_count_1000m,schools_count_2000m,green_share_500m,green_share_1000m,green_share_2000m,water_share_500m,water_share_1000m,water_share_2000m,industrial_share_500m,industrial_share_1000m,industrial_share_2000m,cemetery_share_500m,cemetery_share_1000m,cemetery_share_2000m,major_roads_length_500m,major_roads_length_1000m,major_roads_length_2000m,railway_length_500m,railway_length_1000m,railway_length_2000m,power_lines_length_500m,power_lines_length_1000m,power_lines_length_2000m,nearby_complexes_count_500m,nearby_complexes_count_1000m,nearby_complexes_count_2000m
project_id,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,
551,190.247540,30.714609,661.140915,418.341852,4275.658116,421.647674,159.154735,225.802843,10,30,72,0.105312,0.223349,0.164918,0.000000,0.111986,0.125104,0.006945,0.006512,0.088559,0.0,0.0,0.000000,10070.552928,19038.209902,33517.469159,1633.714755,3715.342768,15163.103916,0.000000,0.000000,0.000000,0,1,11
676,88.324525,62.800377,273.029962,313.759172,3815.389005,313.759172,476.857707,828.813657,11,33,86,0.140036,0.119793,0.208946,0.000000,0.000000,0.092523,0.000048,0.008604,0.091934,0.0,0.0,0.000000,655.987379,14117.019478,30425.794551,0.000000,2835.141365,11543.737055,0.000000,0.000000,0.000000,1,3,12
683,361.790991,308.993178,81.331186,223.651401,3853.174785,1046.170695,940.523005,1435.388139,2,2,5,0.260974,0.087225,0.071654,0.005868,0.005419,0.005649,0.028414,0.061489,0.105868,0.0,0.0,0.000000,0.000000,508.472741,11067.608115,0.000000,0.000000,14993.814115,0.000000,0.000000,0.000000,0,0,1
688,98.273968,107.512755,369.183725,246.960404,2349.284559,357.399015,474.290722,1356.008722,4,14,25,0.055673,0.212474,0.343713,0.011617,0.012290,0.007083,0.045839,0.059854,0.050422,0.0,0.0,0.000000,325.420219,3184.444580,35081.739787,0.000000,0.000000,18238.885551,937.405965,3412.747056,7349.619076,1,2,5
696,114.080661,103.104423,812.219817,252.465171,1801.262631,252.465171,476.177470,2522.634618,6,16,22,0.135891,0.339055,0.554714,0.000000,0.001791,0.004117,0.000297,0.021398,0.021169,0.0,0.0,0.002287,194.866970,3592.102116,22141.559358,0.000000,0.000000,0.000000,0.000000,2048.751567,3395.762762,0,0,1



Описательная статистика:


,mean,min,50%,max
dist_to_metro_m,373.28,9.34,212.45,7465.97
dist_to_school_m,312.93,0.00,194.72,5153.65
dist_to_water_m,361.98,1.00,305.69,1504.57
dist_to_industrial_m,170.74,0.00,114.12,1677.27
dist_to_cemetery_m,1771.08,38.91,1647.77,5564.23
dist_to_power_line_m,278.14,0.32,198.27,1857.18
dist_to_major_road_m,401.76,6.36,283.22,3693.77
dist_to_railway_m,979.99,1.30,486.03,12009.48
schools_count_500m,3.92,0.00,3.00,19.00
schools_count_1000m,13.22,0.00,12.00,46.00


## 4. Сохранение

`geo_features.parquet` индексирован по `project_id`; в `04_features` мерджится
в модельный датасет.

In [22]:
GEO_FEATURES_PARQUET.parent.mkdir(parents=True, exist_ok=True)
geo_feats.to_parquet(GEO_FEATURES_PARQUET)
print(f"Сохранено: {GEO_FEATURES_PARQUET}")
print(f"  {geo_feats.shape[0]} проектов x {geo_feats.shape[1]} геофич")
print("\nNaN% по геофичам:")
display((100 * geo_feats.isna().mean()).round(1).sort_values(ascending=False).head(10).to_frame("NaN%"))

Сохранено: /Users/adesayan/demand_elasticity/data/processed/geo_features.parquet
  752 проектов x 35 геофич

NaN% по геофичам:


,NaN%
dist_to_metro_m,0.0
railway_length_500m,0.0
cemetery_share_500m,0.0
cemetery_share_1000m,0.0
cemetery_share_2000m,0.0
major_roads_length_500m,0.0
major_roads_length_1000m,0.0
major_roads_length_2000m,0.0
railway_length_1000m,0.0
industrial_share_1000m,0.0
